# Can we predict 30-day readmission (`readmit_30d`)?

Plan: inspect the data, then evaluate models with **person-grouped** cross-validation. The file has ~8 stays per person, so a random row split would put the same person in train and test and could overstate performance. I check that explicitly.

In [ ]:
import pandas as pd, numpy as np, matplotlib.pyplot as plt
from sklearn.model_selection import StratifiedGroupKFold, KFold, cross_val_predict
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.dummy import DummyClassifier
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.metrics import roc_auc_score, average_precision_score, brier_score_loss, roc_curve, precision_recall_curve
from sklearn.inspection import permutation_importance
pd.set_option('display.width',140)
df=pd.read_csv('data.csv'); print(pd.read_csv('columns.csv').to_string(index=False))
df.head()

## 1. Data checks

In [ ]:
print(df.shape, '| missing cells:', df.isna().sum().sum(), '| duplicate rows:', df.duplicated().sum())
print('persons:', df.person_key.nunique(), '| stays/person (min, median, max):', df.groupby('person_key').size().describe()[['min','50%','max']].tolist())
print('readmission rate:', df.readmit_30d.mean().round(4))
df.describe().T

In [ ]:
# Is readmission clustered within people? Compare spread of per-person rates to what independent draws would give.
g=df.groupby('person_key').readmit_30d.agg(['mean','size'])
p=df.readmit_30d.mean()
print('observed var of person rates :', round(g['mean'].var(),4))
print('expected if independent      :', round((p*(1-p)/g['size']).mean(),4))

If person rates vary more than chance, stays from one person are not independent, which is why grouping matters.

In [ ]:
df['sex_M']=(df.sex=='M').astype(int)
feats=['visit_number','birth_year','sex_M','postcode_area','bmi','length_of_stay_days','n_medications','lab_a','lab_b']
num=[f for f in feats if f!='sex_M']
fig,ax=plt.subplots(2,4,figsize=(16,7))
for a,f in zip(ax.ravel(),num):
    q=pd.qcut(df[f],10,duplicates='drop'); r=df.groupby(q,observed=True).readmit_30d.mean()
    a.plot(range(len(r)),r.values,'o-'); a.axhline(p,ls='--',c='grey'); a.set_title(f); a.set_xlabel('decile'); a.set_ylabel('readmit rate')
ax.ravel()[-1].axis('off'); plt.tight_layout(); plt.show()
print(df.groupby('sex').readmit_30d.mean())

## 2. Models with person-grouped CV
5-fold `StratifiedGroupKFold` by `person_key`; `person_key` is never a feature. Metrics are pooled out-of-fold: ROC AUC and average precision (a no-skill AP equals prevalence).

In [ ]:
X=df[feats]; y=df.readmit_30d; groups=df.person_key
cv=StratifiedGroupKFold(5,shuffle=True,random_state=0)
pre=ColumnTransformer([('num',StandardScaler(),num)],remainder='passthrough')
models={
 'baseline (prevalence)':DummyClassifier(strategy='prior'),
 'logistic regression':make_pipeline(pre,LogisticRegression(max_iter=1000)),
 'gradient boosting':HistGradientBoostingClassifier(max_depth=3,learning_rate=0.05,max_iter=200,random_state=0),
}
oof={}; rows=[]
for n,m in models.items():
    pr=cross_val_predict(m,X,y,cv=cv,groups=groups,method='predict_proba')[:,1]; oof[n]=pr
    fa=[roc_auc_score(y.iloc[te],pr[te]) for _,te in cv.split(X,y,groups)]
    rows.append(dict(model=n,AUC=roc_auc_score(y,pr),AUC_fold_sd=np.std(fa),AP=average_precision_score(y,pr),Brier=brier_score_loss(y,pr)))
res=pd.DataFrame(rows).set_index('model').round(4); print('prevalence =',round(y.mean(),4)); res

In [ ]:
fig,ax=plt.subplots(1,2,figsize=(11,4))
for n,pr in oof.items():
    f,t,_=roc_curve(y,pr); ax[0].plot(f,t,label=n)
    pp,rr,_=precision_recall_curve(y,pr); ax[1].plot(rr,pp,label=n)
ax[0].plot([0,1],[0,1],'k:'); ax[0].set(xlabel='FPR',ylabel='TPR',title='ROC (grouped OOF)'); ax[1].set(xlabel='recall',ylabel='precision',title='Precision-recall'); ax[0].legend(); plt.tight_layout(); plt.show()

## 3. Leakage check: random row split vs grouped split

In [ ]:
m=models['gradient boosting']
for name,cvx,kw in [('random rows (leaky)',KFold(5,shuffle=True,random_state=0),{}),('grouped by person',cv,{'groups':groups})]:
    pr=cross_val_predict(m,X,y,cv=cvx,method='predict_proba',**kw)[:,1]
    print(f'{name:22s} AUC={roc_auc_score(y,pr):.4f}  AP={average_precision_score(y,pr):.4f}')
Xid=df[feats+['person_key']]
pr=cross_val_predict(m,Xid,y,cv=KFold(5,shuffle=True,random_state=0),method='predict_proba')[:,1]
print(f'{"random + person_key":22s} AUC={roc_auc_score(y,pr):.4f}  (illustration only: lets the model memorise people)')

## 4. What drives the prediction?

In [ ]:
lr=models['logistic regression'].fit(X,y)
coef=pd.Series(lr[-1].coef_[0],index=num+['sex_M']).sort_values(); print('Logistic coefficients (per 1 SD):'); print(coef.round(3))
imp=np.zeros(len(feats))
for tr,te in cv.split(X,y,groups):
    mm=HistGradientBoostingClassifier(max_depth=3,learning_rate=0.05,max_iter=200,random_state=0).fit(X.iloc[tr],y.iloc[tr])
    imp+=permutation_importance(mm,X.iloc[te],y.iloc[te],scoring='roc_auc',n_repeats=5,random_state=0).importances_mean/5
pd.Series(imp,index=feats).sort_values().plot.barh(title='Permutation importance (drop in AUC, held-out)'); plt.show()

## 5. Calibration and lift

In [ ]:
pr=oof['logistic regression']
cal=pd.DataFrame({'pred':pr,'obs':y,'b':pd.qcut(pr,10,labels=False)}).groupby('b').mean()
plt.plot(cal.pred,cal.obs,'o-'); plt.plot([0,.6],[0,.6],'k:'); plt.xlabel('mean predicted'); plt.ylabel('observed'); plt.title('Calibration (logistic, decile bins)'); plt.show()
for k in [0.1,0.2,0.3]:
    fl=pr>=np.quantile(pr,1-k)
    print(f'top {int(k*100)}% flagged: readmit rate {y[fl].mean():.3f} vs {y.mean():.3f} overall; captures {y[fl].sum()/y.sum():.1%} of readmissions')

## 6. Prior-history feature (uses only earlier stays)
Because readmission clusters within people, a person's *earlier* outcomes may be informative. For each stay I compute the person's readmission rate and count over strictly earlier `visit_number`s (NaN for the first stay). This is usable at prediction time only if earlier stays' outcomes are known when the current stay ends; the 30-day outcome of the immediately preceding stay should be known by then, but check this in your real workflow.

In [ ]:
d=df.sort_values(['person_key','visit_number']).copy()
gp=d.groupby('person_key').readmit_30d
d['prior_n']=gp.cumcount()
d['prior_readmits']=gp.cumsum()-d.readmit_30d
d['prior_rate']=d.prior_readmits/d.prior_n.replace(0,np.nan)
d=d.loc[df.index]  # restore original order
feats2=feats+['prior_n','prior_rate']
X2=d[feats2]
gb=HistGradientBoostingClassifier(max_depth=3,learning_rate=0.05,max_iter=200,random_state=0)
pr2=cross_val_predict(gb,X2,y,cv=cv,groups=groups,method='predict_proba')[:,1]
fa=[roc_auc_score(y.iloc[te],pr2[te]) for _,te in cv.split(X2,y,groups)]
print(f'GB + prior history: AUC={roc_auc_score(y,pr2):.4f} (fold sd {np.std(fa):.4f})  AP={average_precision_score(y,pr2):.4f}  Brier={brier_score_loss(y,pr2):.4f}')
print('compare GB without it:'); print(res.loc['gradient boosting'].to_dict())

## 7. Conclusions (from the run on this data, 5-fold grouped CV)
- **Stay-level features alone give weak-to-modest signal**: AUC about 0.65 (logistic) / 0.63 (gradient boosting) vs 0.50 baseline; AP 0.32 vs 0.20 prevalence. Flagging the top 20% by risk finds readmissions at ~34% vs 20% overall (captures ~35%). Not enough for confident individual-level decisions.
- **Drivers**: longer stay, more discharge medications and higher `lab_a`. Other features show little effect. Logistic regression beats boosting, so the signal is roughly linear.
- **Random row splits overstate performance** (AUC 0.68 vs 0.63 grouped), because the same person appears in train and test. Use grouped evaluation.
- **Readmission is strongly person-clustered** (variance of person rates 0.071 vs 0.023 if independent). Adding the person's prior readmission history (earlier stays only) lifts AUC to about 0.81. Confirm that this information is actually available at the time you would score a stay, and that `visit_number` reflects true chronology, before relying on it.
- Caveats: one dataset, no external or time-based validation, no hyperparameter tuning, and model scores are not calibrated for a different population.